# Vilex — Kaggle Stage 5 Only (OmniVoice TTS)

**Repo:** https://github.com/thaiphu05/Vilex @ `dev-thai`  
**Input:** Stage 1-4 JSONs chạy local → upload Kaggle Dataset  
**Output:** `outputs/vi_audio/**/dialogue.wav` (stereo ch0=assistant ch1=user) + `meta.json`

## Kaggle Settings (do once)
- `Settings → Internet ON` (cần cho `pip install git+OmniVoice`)
- `Settings → Accelerator → GPU T4 x2`
- `Add Input`: mount 2 datasets — (1) dialogues Phase1 (`outputs/vi_tt_bc`), (2) voice-clone
- Không cần `Secrets` (Stage5 không gọi Gemini)
- Không cần Restart kernel (single-phase, 1 env duy nhất)


In [ ]:
# Cell 1 — Config (sửa ở đây)
BRANCH = "dev-thai"
REPO = "https://github.com/thaiphu05/Vilex.git"
VOICE_POOL = "/kaggle/input/datasets/thitrnhph/voice-clone"
DIALOGUES_ROOT = "/kaggle/input/vilex-phase1-dialogues"  # slug placeholder — code dưới tự dò nếu sai
NUM_VARIANTS = 1
MAX_DIALOGUES = 1  # smoke test; 0 hoặc bỏ flag = full
DEVICE = "cuda"  # OOM thì đổi "cpu"
print(f"BRANCH={BRANCH} VOICE_POOL={VOICE_POOL} DIALOGUES_ROOT={DIALOGUES_ROOT}")

In [ ]:
# Cell 2 — Clone code (nhẹ, data nặng đi qua Dataset)
!git clone -b $BRANCH $REPO
%cd Vilex
!git branch --show-current && git log --oneline -3
!ls -lh

In [ ]:
# Cell 3 — Install Stage 5 deps (OmniVoice)
# torch đã có sẵn trong Kaggle image; cài shared Block A + OmniVoice Block C (requirements-stage5.txt)
!pip install -q -r requirements-stage5.txt
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q pyloudnorm
!python -c "import torch; print(f'torch={torch.__version__} cuda={torch.cuda.is_available()}')"
!python -c "import whisperx, silero_vad; print('whisperx+silero ok')"

In [ ]:
# Cell 4 — Validate voice pool + dialogues input
# Mỗi *.wav phải có sidecar *.txt transcript chính xác (tts_render/convert_spoken.py:715), cần >=2 cặp.
import os, glob
VOICE_POOL = "/kaggle/input/datasets/thitrnhph/voice-clone"
DIALOGUES_ROOT = "/kaggle/input/vilex-phase1-dialogues"
# Fallback nếu slug mount khác tên dự kiến
if not os.path.isdir(VOICE_POOL):
    for cand in ["/kaggle/input/voice-clone", "voice_clone", "/kaggle/working/voice_clone"]:
        if os.path.isdir(cand):
            VOICE_POOL = cand
            break
print(f"VOICE_POOL={VOICE_POOL} exists={os.path.isdir(VOICE_POOL)}")
!ls -lh "$VOICE_POOL" 2>&1 | head -40
wavs = sorted(glob.glob(os.path.join(VOICE_POOL, "*.wav")))
print(f"Found {len(wavs)} wavs")
for w in wavs[:10]:
    txt = os.path.splitext(w)[0] + ".txt"
    ok = "OK" if os.path.isfile(txt) else "MISSING .txt"
    print(f"{ok} {w}")
if len(wavs) < 2:
    print("ERROR: voice pool needs >=2 wavs with sidecar .txt (convert_spoken.py:732)")
# Dialogues: tự dò nếu DIALOGUES_ROOT placeholder sai
import glob as _g
cands = _g.glob(os.path.join(DIALOGUES_ROOT, "**", "*.json"), recursive=True)
if not cands:
    for root in sorted(_g.glob("/kaggle/input/*")):
        hit = _g.glob(os.path.join(root, "**", "*.json"), recursive=True)
        if hit:
            DIALOGUES_ROOT = root
            cands = hit
            break
print(f"DIALOGUES_ROOT={DIALOGUES_ROOT} jsons={len(cands)}")
for p in cands[:5]:
    print(p)

In [ ]:
# Cell 5 — Stage 5: OmniVoice render (VI default)
# Flags theo tts_render/convert_spoken.py:1550-1635 (--input_glob nargs=+, --save_dir, --num_variants, --device, --max_dialogues).
# Layout input: <root>/text_dialogue_<dataset>/<split>/*.json (Stage 4 output).
VOICE_POOL = "/kaggle/input/datasets/thitrnhph/voice-clone"
import os
for cand in [VOICE_POOL, "/kaggle/input/voice-clone", "voice_clone"]:
    if os.path.isdir(cand):
        VOICE_POOL = cand
        break
print(f"Using VOICE_POOL={VOICE_POOL}")
!python tts_render/convert_spoken.py \
  --tts_backend omnivoice \
  --language vi \
  --input_glob "/kaggle/input/vilex-phase1-dialogues/**/*.json" \
  --save_dir outputs/vi_audio \
  --omnivoice_voice_pool "$VOICE_POOL" \
  --num_variants 1 \
  --device cuda \
  --max_dialogues 1

In [ ]:
# Cell 5b — Nếu CUDA OOM, chạy lại cell này trên CPU (bỏ comment)
# !python tts_render/convert_spoken.py --tts_backend omnivoice --language vi --input_glob "/kaggle/input/vilex-phase1-dialogues/**/*.json" --save_dir outputs/vi_audio --omnivoice_voice_pool "$VOICE_POOL" --num_variants 1 --device cpu --max_dialogues 1

In [ ]:
# Cell 6 — Preview + export
!find outputs/vi_audio -type f 2>/dev/null | head -30
!ls -lh outputs/vi_audio/text_dialogue_interviewer/train/*/var00/dialogues/dialogue.wav 2>/dev/null | head -5
try:
    from IPython.display import Audio, display
    import glob as _g
    wavs = _g.glob("outputs/vi_audio/**/dialogue.wav", recursive=True)
    if wavs:
        print(f"Preview: {wavs[0]}")
        display(Audio(wavs[0]))
    else:
        print("No dialogue.wav yet — check logs above")
except Exception as e:
    print(e)
!zip -r /kaggle/working/vi_audio.zip outputs/vi_audio 2>&1 | tail -20
!ls -lh /kaggle/working/vi_audio.zip 2>/dev/null

## Notes
- **Local export Stage 1-4:** `zip -r vilex-phase1-dialogues.zip outputs/vi_tt_bc -x "*.DS_Store"` rồi `Kaggle → Datasets → New` — không push outputs lên GitHub (`outputs/` bị ignore).
- **Full run:** bỏ `--max_dialogues 1` ở Cell 5 và tăng `--num_variants`. Re-run resume tự skip variant đã có (`dialogue.wav` + `meta.json` tồn tại).
- **Output layout:** `outputs/vi_audio/text_dialogue_<ds>/<split>/<id>/varNN/dialogues/dialogue.wav` + `meta.json` (xem `docs/stage5-tts.md:38`).
- **Persist:** `/kaggle/working/vi_audio.zip` → `Save Version → Output`.
